# 01. Data analysis and empirical coarse-graining diagnostics

This notebook reads `data/SAMPLE.csv`, validates the monthly multi-sector schema, and computes empirical temporal coarse-graining diagnostics.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import binom, betabinom, norm, rankdata, spearmanr, kendalltau
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(".")
DATA = ROOT / "data"
PDATA = ROOT / "pdata"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
for directory in [DATA, PDATA, FIGURES, TABLES]:
    directory.mkdir(exist_ok=True)

RNG = np.random.default_rng(20260618)
plt.style.use("default")


SECTOR_COLORS = {
    "Basic Materials": "#4C78A8",
    "Capital Goods": "#F58518",
    "Consumer": "#54A24B",
    "Energy": "#E45756",
    "Finance": "#72B7B2",
    "Healthcare": "#B279A2",
    "Technology": "#FF9DA6",
    "Utilities": "#9D755D",
}


def read_sample():
    df = pd.read_csv(DATA / "SAMPLE.csv", parse_dates=["date"])
    expected = ["date", "sector", "obligors", "defaulted"]
    if list(df.columns) != expected:
        raise ValueError(f"Expected columns {expected}, got {list(df.columns)}")
    df = df.sort_values(["date", "sector"]).reset_index(drop=True)
    if (df["obligors"] <= 0).any():
        raise ValueError("obligors must be positive")
    if ((df["defaulted"] < 0) | (df["defaulted"] > df["obligors"])).any():
        raise ValueError("defaulted must lie between zero and obligors")
    return df


def empirical_logit(defaulted, obligors, smooth=0.5):
    defaulted = np.asarray(defaulted, dtype=float)
    obligors = np.asarray(obligors, dtype=float)
    return np.log((defaulted + smooth) / (obligors - defaulted + smooth))


def panel_matrices(df):
    dates = pd.Index(sorted(df["date"].unique()), name="date")
    sectors = pd.Index(sorted(df["sector"].unique()), name="sector")
    dmat = (
        df.pivot(index="date", columns="sector", values="defaulted")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    nmat = (
        df.pivot(index="date", columns="sector", values="obligors")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    x = pd.DataFrame(
        empirical_logit(dmat.to_numpy(), nmat.to_numpy()),
        index=dates,
        columns=sectors,
    )
    rates = dmat / nmat
    return dates, sectors, dmat, nmat, x, rates


def aggregate_k_month(df, k):
    wide_dates = pd.Index(sorted(df["date"].unique()))
    block_lookup = {date: i // k for i, date in enumerate(wide_dates)}
    tmp = df.copy()
    tmp["block"] = tmp["date"].map(block_lookup)
    tmp["block_start"] = tmp["block"].map(lambda b: wide_dates[b * k])
    out = (
        tmp.groupby(["block", "block_start", "sector"], as_index=False)
        .agg(obligors=("obligors", "sum"), defaulted=("defaulted", "sum"))
    )
    out = out.rename(columns={"block_start": "date"})
    out["k_month"] = k
    return out[["date", "sector", "k_month", "obligors", "defaulted"]]


def fit_common_eps_model(df, R=2):
    dates, sectors, dmat, nmat, x, rates = panel_matrices(df)
    X = x.to_numpy()
    intercept = X.mean(axis=0)
    Xc = X - intercept
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    loadings = eigvecs[:, :R]
    scores = Xc @ loadings
    phis = []
    innov_sd = []
    for r in range(R):
        y = scores[1:, r]
        z = scores[:-1, r]
        denom = float(np.dot(z, z))
        phi = float(np.dot(z, y) / denom) if denom > 1e-12 else 0.0
        phi = float(np.clip(phi, -0.98, 0.98))
        resid = y - phi * z
        phis.append(phi)
        innov_sd.append(float(np.sqrt(np.mean(resid**2) + 1e-8)))
    fitted_centered = scores @ loadings.T
    residual = Xc - fitted_centered
    sigma_eps_common = float(np.sqrt(np.mean(residual**2) + 1e-8))
    eta_hat = intercept + fitted_centered
    p_hat = expit(eta_hat)
    ll = float(binom.logpmf(dmat.to_numpy(), nmat.to_numpy(), p_hat).sum())
    n_params = len(sectors) + R * (len(sectors) + 2) + 1
    aic = float(2 * n_params - 2 * ll)
    return {
        "dates": list(dates),
        "sectors": list(sectors),
        "defaulted": dmat,
        "obligors": nmat,
        "logit": x,
        "rates": rates,
        "intercept": intercept,
        "eigvals": eigvals,
        "loadings": loadings,
        "scores": scores,
        "phis": np.array(phis),
        "innov_sd": np.array(innov_sd),
        "sigma_eps_common": sigma_eps_common,
        "eta_hat": eta_hat,
        "p_hat": p_hat,
        "log_likelihood": ll,
        "aic": aic,
    }


def acf_values(x, max_lag=24):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.dot(x, x)
    if denom <= 1e-12:
        return np.zeros(max_lag + 1)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.array(vals)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

In [ ]:
df = read_sample()
df["default_rate"] = df["defaulted"] / df["obligors"]
table1 = (
    df.groupby("sector")
    .agg(
        months=("date", "nunique"),
        obligor_months=("obligors", "sum"),
        defaults=("defaulted", "sum"),
        mean_monthly_default_rate=("default_rate", "mean"),
        sd_monthly_default_rate=("default_rate", "std"),
    )
    .reset_index()
)
table1.loc[len(table1)] = [
    "All sectors",
    df["date"].nunique(),
    df["obligors"].sum(),
    df["defaulted"].sum(),
    df["defaulted"].sum() / df["obligors"].sum(),
    df["default_rate"].std(),
]
table1.to_csv(TABLES / "table1_data_construction_sector_summary.csv", index=False)
table1

In [ ]:
rows = []
aggregated = []
for k in [1, 3, 6, 12]:
    agg = aggregate_k_month(df, k)
    agg["monthly_equivalent_default_rate"] = agg["defaulted"] / agg["obligors"]
    aggregated.append(agg)
    panel = agg.pivot(index="date", columns="sector", values="monthly_equivalent_default_rate")
    corr = panel.corr()
    eigvals = np.linalg.eigvalsh(corr.fillna(0.0).to_numpy())[::-1]
    for sector, g in agg.groupby("sector"):
        rows.append({
            "k_month": k,
            "sector": sector,
            "blocks": len(g),
            "mean_monthly_equivalent_default_rate": g["monthly_equivalent_default_rate"].mean(),
            "var_monthly_equivalent_default_rate": g["monthly_equivalent_default_rate"].var(ddof=1),
            "mean_pairwise_correlation": corr.where(~np.eye(len(corr), dtype=bool)).stack().mean(),
            "leading_correlation_eigenvalue_share": eigvals[0] / eigvals.sum(),
        })
coarse = pd.DataFrame(rows)
coarse.to_csv(PDATA / "coarse_graining_summary.csv", index=False)
pd.concat(aggregated, ignore_index=True).to_csv(PDATA / "coarse_grained_default_counts.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5))
for sector, g in coarse.groupby("sector"):
    axes[0].plot(g["k_month"], g["var_monthly_equivalent_default_rate"], marker="o", lw=1.2, label=sector)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xticks([1, 3, 6, 12])
axes[0].set_xticklabels(["1", "3", "6", "12"])
axes[0].set_xlabel("Aggregation scale k, months")
axes[0].set_ylabel("Variance")
axes[0].set_title("Variance scaling")

scale_summary = coarse.groupby("k_month", as_index=False).agg(
    mean_pairwise_correlation=("mean_pairwise_correlation", "mean"),
    leading_correlation_eigenvalue_share=("leading_correlation_eigenvalue_share", "mean"),
)
axes[1].plot(scale_summary["k_month"], scale_summary["mean_pairwise_correlation"], marker="o", label="Mean pairwise correlation")
axes[1].plot(scale_summary["k_month"], scale_summary["leading_correlation_eigenvalue_share"], marker="s", label="Leading eigenvalue share")
axes[1].set_xscale("log")
axes[1].set_xticks([1, 3, 6, 12])
axes[1].set_xticklabels(["1", "3", "6", "12"])
axes[1].set_xlabel("Aggregation scale k, months")
axes[1].set_title("Correlation/eigenvalue scaling")
axes[1].legend(fontsize=8)
fig.suptitle("Figure 1. Empirical variance scaling and eigenvalue scaling")
savefig(FIGURES / "figure1_empirical_variance_eigenvalue_scaling.png")
coarse.head()

In [ ]:
quarterly = aggregate_k_month(df, 3)
quarterly["default_rate"] = quarterly["defaulted"] / quarterly["obligors"]
heat = quarterly.pivot(index="date", columns="sector", values="default_rate")

fig, ax = plt.subplots(figsize=(10.5, 4.8))
im = ax.imshow(heat.T, aspect="auto", cmap="viridis")
ax.set_yticks(np.arange(len(heat.columns)))
ax.set_yticklabels(heat.columns, fontsize=8)
tick_pos = np.linspace(0, len(heat.index) - 1, 8, dtype=int)
ax.set_xticks(tick_pos)
ax.set_xticklabels([pd.Timestamp(heat.index[i]).strftime("%Y") for i in tick_pos], rotation=0)
ax.set_xlabel("Quarter")
ax.set_title("Figure 10. Quarterly sector default-rate heatmap")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="Default rate")
savefig(FIGURES / "figure10_quarterly_sector_default_rate_heatmap.png")